# AdaptMem × MemPal Matched-Protocol Benchmark

Run mempal's `longmemeval_bench.py` on Colab with adaptmem's FT-300 encoder substituted in. Produces matched-protocol numbers comparable to mempal's published raw / hybrid_v4 results.

**Three runs (ardışık):**
1. **Sanity:** mempal raw + ChromaDB default encoder → expected R@5 ~0.966 (env doğrulama)
2. **Matched main claim:** mempal raw + FT-300 SBERT → adaptmem fine-tune'ın matched-protocol R@5'i
3. **Orthogonal lift:** mempal hybrid_v4 + FT-300 SBERT → FT, mempal'ın en iyi mode'u üstüne lift veriyor mu

**Drive layout beklenen:**
```
/content/drive/MyDrive/adaptmem-bench/
  ├── minilm-lme-ft-300/         ← FT-300 sentence-transformers model dir (90MB)
  └── longmemeval_s_cleaned.json ← dataset (265MB)
```

**Suggested runtime:** CPU yeterli (chromadb default + SBERT inference). T4 hızlandırır ama gerekli değil.


## 1. Setup — install deps + clone repos

In [16]:
!pip install -q chromadb sentence-transformers fastembed

In [17]:
%cd /content
!git clone --depth 1 https://github.com/MemPalace/mempalace.git mempalace_repo
!git clone --depth 1 https://github.com/nakata-app/adaptmem.git
!pip install -q -e ./mempalace_repo  # mempal package (for hybrid_v4 mode)

/content
fatal: destination path 'mempalace_repo' already exists and is not an empty directory.
fatal: destination path 'adaptmem' already exists and is not an empty directory.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mempalace (pyproject.toml) ... done


## 2. Mount Drive — model + dataset

In [18]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/adaptmem-bench'
FT_MODEL  = f'{DRIVE_DIR}/minilm-lme-ft-300/model/model'
DATASET   = f'{DRIVE_DIR}/longmemeval_s_cleaned.json'

assert os.path.isdir(FT_MODEL),  f'FT model not found at {FT_MODEL}'
assert os.path.isfile(DATASET),  f'Dataset not found at {DATASET}'
print(f'FT model:  {FT_MODEL}')
print(f'Dataset:   {DATASET}  ({os.path.getsize(DATASET) / 1024**2:.0f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FT model:  /content/drive/MyDrive/adaptmem-bench/minilm-lme-ft-300/model/model
Dataset:   /content/drive/MyDrive/adaptmem-bench/longmemeval_s_cleaned.json  (265 MB)


## 3. Path setup

In [19]:
BENCH_SCRIPT = '/content/mempalace_repo/benchmarks/longmemeval_bench.py'
WRAPPER      = '/content/adaptmem/benchmarks/mempal_bench_with_ft.py'
OUT_DIR      = '/content/results'
import os
os.makedirs(OUT_DIR, exist_ok=True)

# Verify
for p in [BENCH_SCRIPT, WRAPPER]:
    assert os.path.isfile(p), f'Missing: {p}'
print('All paths OK.')

All paths OK.


## 4. Run #0 — Sanity: mempal raw + ChromaDB default encoder

Beklenen: R@5 ≈ 0.966 (mempal'ın yayınladığı raw rakam).

In [15]:
!python {WRAPPER} \
    --bench-script {BENCH_SCRIPT} \
    --data-file    {DATASET} \
    --mode         raw \
    --out          {OUT_DIR}/run0_raw_default.jsonl
# (no --ft-model: falls through to mempal's default encoder)

  [sanity] Using mempal's ChromaDB default encoder.

  MemPal × LongMemEval Benchmark
  Data:        longmemeval_s_cleaned.json
  Questions:   500
  Granularity: session
  Mode:        raw
────────────────────────────────────────────────────────────

  [   1/500] e47becba                       R@5=0 R@10=1  miss
  [   2/500] 118b2229                       R@5=1 R@10=1  HIT
  [   3/500] 51a45a95                       R@5=1 R@10=1  HIT
  [   4/500] 58bf7951                       R@5=1 R@10=1  HIT
  [   5/500] 1e043500                       R@5=1 R@10=1  HIT
  [   6/500] c5e8278d                       R@5=1 R@10=1  HIT
  [   7/500] 6ade9755                       R@5=1 R@10=1  HIT
  [   8/500] 6f9b354f                       R@5=1 R@10=1  HIT
  [   9/500] 58ef2f1c                       R@5=1 R@10=1  HIT
  [  10/500] f8c5f88b                       R@5=1 R@10=1  HIT
  [  11/500] 5d3d2817                       R@5=0 R@10=0  miss
  [  12/500] 7527f7e2                       R@5=1 R@10=1  HIT
  [

## 5. Run #1 — Matched main claim: mempal raw + FT-300 SBERT

In [20]:
!python {WRAPPER} \
    --bench-script {BENCH_SCRIPT} \
    --data-file    {DATASET} \
    --ft-model     {FT_MODEL} \
    --mode         raw \
    --out          {OUT_DIR}/run1_raw_ft300.jsonl

  Loading sentence-transformers model: /content/drive/MyDrive/adaptmem-bench/minilm-lme-ft-300/model/model
Loading weights: 100% 103/103 [00:02<00:00, 37.84it/s, Materializing param=pooler.dense.weight]
  Model ready.
  [matched-protocol] Substituted bench encoder → /content/drive/MyDrive/adaptmem-bench/minilm-lme-ft-300/model/model

  MemPal × LongMemEval Benchmark
  Data:        longmemeval_s_cleaned.json
  Questions:   500
  Granularity: session
  Mode:        raw
────────────────────────────────────────────────────────────

  [   1/500] e47becba                       R@5=1 R@10=1  HIT
  [   2/500] 118b2229                       R@5=1 R@10=1  HIT
  [   3/500] 51a45a95                       R@5=1 R@10=1  HIT
  [   4/500] 58bf7951                       R@5=1 R@10=1  HIT
  [   5/500] 1e043500                       R@5=1 R@10=1  HIT
  [   6/500] c5e8278d                       R@5=1 R@10=1  HIT
  [   7/500] 6ade9755                       R@5=1 R@10=1  HIT
  [   8/500] 6f9b354f           

## 6. Run #2 — Orthogonal lift: mempal hybrid_v4 + FT-300 SBERT

In [21]:
!python {WRAPPER} \
    --bench-script {BENCH_SCRIPT} \
    --data-file    {DATASET} \
    --ft-model     {FT_MODEL} \
    --mode         hybrid_v4 \
    --out          {OUT_DIR}/run2_hybrid_v4_ft300.jsonl

  Loading sentence-transformers model: /content/drive/MyDrive/adaptmem-bench/minilm-lme-ft-300/model/model
Loading weights: 100% 103/103 [00:00<00:00, 394.27it/s, Materializing param=pooler.dense.weight]
  Model ready.
  [matched-protocol] Substituted bench encoder → /content/drive/MyDrive/adaptmem-bench/minilm-lme-ft-300/model/model

  MemPal × LongMemEval Benchmark
  Data:        longmemeval_s_cleaned.json
  Questions:   500
  Granularity: session
  Mode:        hybrid_v4
────────────────────────────────────────────────────────────

  [   1/500] e47becba                       R@5=1 R@10=1  HIT
  [   2/500] 118b2229                       R@5=1 R@10=1  HIT
  [   3/500] 51a45a95                       R@5=1 R@10=1  HIT
  [   4/500] 58bf7951                       R@5=1 R@10=1  HIT
  [   5/500] 1e043500                       R@5=1 R@10=1  HIT
  [   6/500] c5e8278d                       R@5=1 R@10=1  HIT
  [   7/500] 6ade9755                       R@5=1 R@10=1  HIT
  [   8/500] 6f9b354f    

## 7. Summary — extract R@1/R@5/R@10 from JSONL outputs

In [22]:
import json, os
from collections import defaultdict

def summarize(jsonl_path):
    if not os.path.isfile(jsonl_path):
        return None
    n = 0
    hits = defaultdict(int)
    with open(jsonl_path) as f:
        for line in f:
            try:
                row = json.loads(line)
            except Exception:
                continue
            # Mempal bench outputs per-question ranking + correct-set; aggregate
            if 'recall' in row:
                for k in (1, 5, 10):
                    if str(k) in row.get('recall', {}):
                        hits[k] += row['recall'][str(k)]
                n += 1
    if n == 0:
        return f'{jsonl_path}: no rows parsed (check schema)'
    return {f'R@{k}': round(hits[k] / n, 4) for k in (1, 5, 10)} | {'n': n}

for label, path in [
    ('Run #0 raw default', f'{OUT_DIR}/run0_raw_default.jsonl'),
    ('Run #1 raw FT-300',  f'{OUT_DIR}/run1_raw_ft300.jsonl'),
    ('Run #2 hybrid_v4 FT-300', f'{OUT_DIR}/run2_hybrid_v4_ft300.jsonl'),
]:
    print(f'{label:30s} → {summarize(path)}')

print('\nNot: jsonl schema mempal repo\'inde değişmiş olabilir; summarize() fonksiyonu çıktı görüldükten sonra düzeltilebilir.')

Run #0 raw default             → /content/results/run0_raw_default.jsonl: no rows parsed (check schema)
Run #1 raw FT-300              → /content/results/run1_raw_ft300.jsonl: no rows parsed (check schema)
Run #2 hybrid_v4 FT-300        → /content/results/run2_hybrid_v4_ft300.jsonl: no rows parsed (check schema)

Not: jsonl schema mempal repo'inde değişmiş olabilir; summarize() fonksiyonu çıktı görüldükten sonra düzeltilebilir.


## 8. Save results back to Drive

In [23]:
import shutil, os
RESULTS_DRIVE = f'{DRIVE_DIR}/results'
os.makedirs(RESULTS_DRIVE, exist_ok=True)
for f in os.listdir(OUT_DIR):
    shutil.copy2(f'{OUT_DIR}/{f}', RESULTS_DRIVE)
print(f'Results copied to {RESULTS_DRIVE}')

Results copied to /content/drive/MyDrive/adaptmem-bench/results
